# Übung 2.10: Abtastung und Rekonstruktion

Ausgehend von einem zeitkontinuierlichen Signal $x_{\mathrm{K}}(t)$ und den Eigenschaften der Fourier-Transformation (2.4.1, 2.4.2) erhalten wir eine Folge $x(k)$ durch Abtastung mit der Frequenz $f_A$ und dem Transformationspaar (2.3.11, 2.3.12). Die Spektren $X_{\mathrm{K}}(j\omega)$ und $X(e^{j\Omega})$ sind durch den Ausdruck (2.4.8) verknüpft:

$$
X(e^{j\Omega})=\frac{1}{T_A}\sum_{i=-\infty}^{\infty} X_{\mathrm{K}}\left(j\frac{\Omega+i2\pi}{T_A}\right)
$$

Somit kann das Spektrum eines diskreten Signals $X(e^{j\Omega})$ durch Überlagerung der Spektren eines zeitkontinuierlichen Signals $X_{\mathrm{K}}(j\omega)$, die um Vielfache von $2\pi$ verschoben und mit $1/T_A$ normiert sind, erhalten werden.

Um das zeitkontinuierliche Signal aus einem diskreten Signal zu rekonstruieren, dürfen sich die Teilspektren $X_{\mathrm{K}}(j(\Omega+i2\pi)/T)$ nicht überlappen[^1]. Daraus ergeben sich die Anforderungen des sogenannten **Abtasttheorems** von Shannon:

[^1]: Der Fall, in dem diese Spektren überlappen, wird *Aliasing* genannt.

* $X_{\mathrm{K}}(j\omega)$ muss bandbegrenzt sein — das heißt, es muss im Intervall $|\omega| \ge \omega_{\text{max}}$ verschwinden.
* Die Abtastfrequenz $\omega_A = 2\pi f_A = 2\pi /T_A$ muss doppelt so groß sein wie die maximale Frequenz $\omega_{\text{max}}$ von $X_{\mathrm{K}}(j\omega)$.

---

Um Abtasteffekte zu untersuchen, ist es notwendig, das zeitkontinuierliche Signal mit einer zeitdiskreten Folge zu simulieren. Dabei werden zwei Abtastperioden verwendet.

Eine Abtastperiode $\Delta t$ wird verwendet, um das kontinuierliche Signal zu simulieren. Die andere Abtastperiode $T_A$ dient zur Abtastung des „analogen“ Signals.  
Die Effekte der Abtastung sind im Frequenzbereich leicht zu erkennen. Zur Simulation der Fourier-Transformation eines kontinuierlichen Signals wird die Funktion `lafplot()` verwendet (siehe `helper_functions.py` für weitere Hilfe).

## a) Simulation des folgenden Signals

$$
s(t) = \cos(2\pi f_0 t + \phi), \quad 0 \leq t \leq T
$$

Die Signalfrequenz soll $f_0 = 2\ \mathrm{kHz}$ betragen und die Abtastfrequenz  
$\frac{1}{\Delta t} = 80\ \mathrm{kHz}$ sein.  
Wähle $T$ so, dass das Signal etwa 2000 Samples des simulierten analogen Signals enthält.

Stelle das „analoge“ Signal mit der Funktion `plt.plot()` dar. Achte dabei auf eine korrekte Skalierung der Zeitachse.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import freqz, cheby2, filtfilt
from helper_functions import lzerofill, lafplot, ldtft

# BEGIN SOLUTION
fsim = 80000                           # Abtastfrequenz
delta_t = np.arange(0, 0.02, 1/fsim)   # Zeitvektor
f0 = 2000                              # Frequenz der Kosinus-Schwingung
s = np.cos(2 * np.pi * f0 * delta_t)   # Kosinussignal

# Plotten
plt.figure()
plt.plot(delta_t, s)
plt.xlabel('Zeit (Sekunden)')
plt.ylabel('Amplitude')
plt.title('Analoges Signal')
plt.show()
# END SOLUTION

## b) Spektrum des „analogen“ Signals darstellen

Verwenden Sie die Funktion `lafplot()`, um das Spektrum darzustellen.

In [ ]:
# BEGIN SOLUTION
lafplot(s, 1/fsim)
# END SOLUTION

## c) Abtastung mit einem A/D-Wandler

Ein A/D-Wandler tastet ein analoges Signal mit der Periode $T_A$ ab.  
Simulieren Sie dieses Verhalten, indem Sie das analoge Signal mit einer Frequenz von $f_A = 8\ \mathrm{kHz}$ abtasten und das Ergebnis darstellen.

In [ ]:
# BEGIN SOLUTION
fa = 8000                          # Abtastfrequenz
l = fsim // fa                     # Downsampling-Faktor
index = np.arange(1, len(s), l)    # Indizes für Abtastung
sd = s[index]                      # (Neu)abtastetes Signal

# Plotten
plt.figure()
plt.stem(sd[:50])
plt.grid()
plt.xlabel('k')
plt.ylabel('Amplitude')
plt.title('Diskretes Signal')
plt.show()
# END SOLUTION

## d) Spektrum des abgetasteten Signals berechnen

Verwenden Sie die Funktion `ldtft()` (siehe `helper_functions.py`).

In [ ]:
# BEGIN SOLUTION
H, W = ldtft(sd, 512)

# Plotten
plt.figure()
plt.plot(W / np.pi, np.abs(H))
plt.grid()
plt.xlabel('Omega / π')
plt.ylabel('Amplitude')
plt.title('Diskrete Fourier-Transformation (Betrag)')
plt.show()
# END SOLUTION

## e) Entwurf eines Tiefpassfilters zur Rekonstruktion des analogen Signals

Dieser analoge Rekonstruktions-Tiefpassfilter muss erneut simuliert werden.  
Verwende die Python-Äquivalenzfunktion (z.B. aus `scipy.signal.cheby2`):

```python
b, a = cheby2(9, 60, fcut)
```

Diese Funktion entwirft einen Tiefpass 9. Ordnung mit einer Sperrbanddämpfung von $60\ \mathrm{dB}$.  
Die analoge Grenzfrequenz `fcut` soll $\frac{1}{2}f_A$ betragen.  
In Python muss diese Frequenz normalisiert werden:  

```
fcut = fsamp / fsim
```

Stellen Sie die Übertragungsfunktion des Tiefpassfilters mit der Funktion `freqz()` dar.


In [ ]:
# BEGIN SOLUTION
fcut = fa / fsim
b, a = cheby2(9, 60, fcut)
W, H = freqz(b, a, 128, fsim)

# Plotten
# Frequenzgang
plt.figure()
plt.semilogy(W, np.abs(H))
plt.grid()
plt.xlabel('Frequenz in Hz')
plt.ylabel('Amplitude')
plt.title('Rekonstruktions-Chebyshev-Tiefpass')
plt.show()

# Phasengang
plt.figure()
plt.plot(W, np.angle(H))
plt.grid()
plt.xlabel('Frequenz in Hz')
plt.ylabel('Phase')
plt.title('Rekonstruktions-Chebyshev-Tiefpass')
plt.show()
# END SOLUTION

## f) Rekonstruktion mit D/A-Wandler

Ein D/A-Wandler erzeugt ein treppenförmiges analoges Zeitsignal.  
Dieses Signal wird durch den entworfenen Chebyshev-Tiefpassfilter geglättet.  
Um diesen Prozess zu simulieren, werden Nullen zwischen die Abtastwerte eingefügt und das Signal anschließend mit dem entworfenen Rekonstruktions-Tiefpassfilter gefiltert.  

Stellen Sie das rekonstruierte analoge Signal sowohl im Zeit- als auch im Frequenzbereich dar.  

**Hinweis:** Die Funktion `lzerofill()` fügt die gewünschte Anzahl von Nullen zwischen die Elemente eines Vektors ein.

In [ ]:
# BEGIN SOLUTION
sa = lzerofill(sd, l)  # Rekonstruktion / Einfügen von Nullen

sa = sa.flatten()
sr = filtfilt(b, a, sa)  # Rekonstruktion

# Plotten
# Rekonstruierte Sinuswelle
plt.figure()
plt.plot(sr)
plt.title('Rekonstruierte Sinuswelle')
plt.xlabel('Zeit in Samples')
plt.ylabel('Amplitude')
plt.show()

# Amplitude
lafplot(sr, 1/fsim)
plt.ylabel('Amplitude')
plt.show()
# END SOLUTION